# EVA Algorithm Artifact Reproduction Notebook (Optional)

This notebook displays the algorithm-level accuracy tables from the EVA paper
(Tables III, IV, and X). It loads pre-computed evaluation results from `algorithm/output/`.

**Note:** The algorithm (AQLM-based quantization) has **not** been specifically optimized for EVA.
The primary contribution of this work is the hardware accelerator design.

**Prerequisites:**
- Run the evaluation commands from `algorithm/README.md` to generate results.
- Results are stored in `algorithm/output/` (git-ignored).

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'simulator').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError(
        'Could not resolve the Eva repository root. '
        'Open this notebook from inside the Eva repository.'
    )


repo_root = resolve_repo_root()
results_dir = repo_root / 'algorithm' / 'output'

display(Markdown(f'Repository root: `{repo_root}`'))
display(Markdown(f'Results directory: `{results_dir}`'))

if not results_dir.exists():
    display(Markdown('**Warning:** Results directory does not exist. '
                     'Run the evaluation commands from `algorithm/README.md` first.'))

## Baseline Data (from published results)

Results for baseline methods are taken directly from their respective publications.
Only the EVA (AQLM) rows are reproduced by running inference.

In [ ]:
# ── TABLE III baselines: WikiText-2 Perplexity ──
TABLE_III_HEADER = {
    'arch':    ['SA',      'ANT',  'FIGNA', 'FIGLUT', 'EVA',  'FIGNA', 'EVA',  'FP16'],
    'algo':    ['QSERVE',  'ANT',  'AWQ',   'BCQ',    'AQLM', 'AWQ',   'AQLM', 'FP16'],
    'fc_act':  ['INT8',    '8bit', 'FP16',  'FP16',   'FP16', 'FP16',  'FP16', 'FP16'],
    'fc_wgt':  ['INT8',    '8bit', 'INT4',  '4bit',   '4bit', 'INT2',  '2bit', 'FP16'],
}

TABLE_III_BASELINES = {
    'L-2 7B':  [5.56, 5.58, 5.60, 5.58, None, 2.2e5, None, 5.12],
    'L-2 13B': [4.95, 5.20, 4.97, 4.96, None, 1.2e5, None, 4.57],
}

# ── TABLE IV baselines: LLaMA-2-7B downstream accuracy ──
TABLE_IV_COLS = ['PIQA', 'COPA', 'ARC-E', 'ARC-C', 'Winogrande']

TABLE_IV_BASELINES = {
    ('FP16', 16):         [78.1, 87.0, 76.4,  43.5,  69.1],
    ('AWQ', 4):           [76.7, 86.0, 53.6,  40.01, 66.3],
    ('GPTQ', 4):          [77.9, 84.5, 69.05, 42.5,  68.3],
    ('LLM.265 FB', 4):    [78.8, None, 73.83, 44.1,  69.7],
    ('LLM.265 VB', 4):    [78.9, None, 73.82, 45,    69.7],
    ('LLM.265 FB', 2):    [54.3, 68.5, 29.76, 30.5,  51.8],
    ('LLM.265 VB', 2):    [56.7, 68.9, 34.52, 31.1,  52.3],
}

# ── TABLE X baselines: MoE downstream accuracy ──
TABLE_X_COLS = ['ARC-C', 'ARC-E', 'PIQA', 'BoolQ', 'Winogrande']

TABLE_X_MIXTRAL_BASELINES = {
    ('FP16', 16):         [59.81, 83.54, 83.73, 85.26, 76.56],
    ('AWQ', 4):           [58.87, 82.58, 82.97, 84.34, 76.24],
    ('AQLM-2x16', 4):    [54.61, 83.12, 81.99, None,  74.82],
    ('GPTQ', 2):          [27.30, 35.44, 59.79, 52.08, 50.83],
    ('GPTVQ-4D', 2):      [46.42, 65.57, 78.13, 78.59, 71.11],
    ('AQLM-1x16', 2):    [47.93, 77.68, 80.43, None,  75.93],
}

TABLE_X_QWEN_BASELINES = {
    ('FP16', 16):         [62.80, 83.75, 80.36, 88.47, 73.88],
    ('AWQ', 4):           [61.43, 82.66, 80.90, 88.87, 73.01],
}

print('Baseline data loaded.')

## Load Results

The cells below load pre-computed evaluation results from `algorithm/output/`.
Missing results will appear as "—" in the tables.

To generate results, run the commands in `algorithm/README.md`.

In [ ]:
def load_ppl_result(name: str):
    """Load a perplexity result JSON, returning None if not found."""
    path = results_dir / f'{name}.json'
    if path.exists():
        data = json.loads(path.read_text())
        print(f'  Loaded {name}: PPL = {data["perplexity"]:.4f}')
        return data
    print(f'  {name}: not found')
    return None


def load_lmeval_result(name: str):
    """Load an lm-eval result JSON, returning None if not found."""
    path = results_dir / name / 'results.json'
    if path.exists():
        data = json.loads(path.read_text())
        tasks = list(data.get('results', {}).keys())
        print(f'  Loaded {name}: tasks = {tasks}')
        return data
    print(f'  {name}: not found')
    return None


def extract_acc(results_json, task_key):
    """Extract accuracy from lm-eval results JSON."""
    try:
        task_data = results_json['results'][task_key]
        acc = task_data.get('acc', task_data.get('acc,none', task_data.get('acc_norm,none')))
        if acc is not None:
            return round(acc * 100, 2) if acc < 1.0 else round(acc, 2)
    except (KeyError, TypeError):
        pass
    return '—'


def build_eva_row(results_json, task_map):
    return [extract_acc(results_json, task_map[col]) for col in task_map]


print(f'Results directory: {results_dir}')

### TABLE III — WikiText-2 Perplexity

In [ ]:
ppl_names = [
    'ppl_llama2_7b_4bit',
    'ppl_llama2_7b_2bit',
    'ppl_llama2_13b_4bit',
    'ppl_llama2_13b_2bit',
]

ppl_results = {}
for name in ppl_names:
    r = load_ppl_result(name)
    if r:
        ppl_results[name] = r['perplexity']

print('\nPerplexity results:', ppl_results)

In [ ]:
# Build TABLE III
eva_4bit_7b  = ppl_results.get('ppl_llama2_7b_4bit',  '—')
eva_2bit_7b  = ppl_results.get('ppl_llama2_7b_2bit',  '—')
eva_4bit_13b = ppl_results.get('ppl_llama2_13b_4bit', '—')
eva_2bit_13b = ppl_results.get('ppl_llama2_13b_2bit', '—')

def fmt_ppl(v):
    if v is None or v == '—':
        return '—'
    if isinstance(v, float) and v > 1000:
        return f'{v:.0e}'
    return f'{v:.2f}' if isinstance(v, float) else str(v)

baselines_7b  = TABLE_III_BASELINES['L-2 7B'].copy()
baselines_13b = TABLE_III_BASELINES['L-2 13B'].copy()
baselines_7b[4]  = eva_4bit_7b
baselines_7b[6]  = eva_2bit_7b
baselines_13b[4] = eva_4bit_13b
baselines_13b[6] = eva_2bit_13b

header = TABLE_III_HEADER
df3 = pd.DataFrame({
    'Arch.':   [''] + header['arch'],
    'Algo.':   [''] + header['algo'],
    'FC Act.': [''] + header['fc_act'],
    'FC Wgt.': [''] + header['fc_wgt'],
    'L-2 7B':  [''] + [fmt_ppl(v) for v in baselines_7b],
    'L-2 13B': [''] + [fmt_ppl(v) for v in baselines_13b],
}).T

display(Markdown('### TABLE III — The Perplexity on WikiText Data'))
display(Markdown('*EVA columns are reproduced; all other values from published results.*'))
display(df3.style.hide(axis='columns'))

### TABLE IV — LLaMA-2-7B Downstream Accuracy

In [ ]:
llama2_downstream_names = ['llama2_7b_4bit', 'llama2_7b_2bit']

llama2_lmeval_results = {}
for name in llama2_downstream_names:
    r = load_lmeval_result(name)
    if r:
        llama2_lmeval_results[name] = r

In [ ]:
TASK_KEY_MAP_IV = {
    'PIQA': 'piqa', 'COPA': 'copa',
    'ARC-E': 'arc_easy', 'ARC-C': 'arc_challenge',
    'Winogrande': 'winogrande',
}

rows_iv = []
for (method, bits), vals in TABLE_IV_BASELINES.items():
    row = {'Method': method, 'Bits': bits}
    for col, v in zip(TABLE_IV_COLS, vals):
        row[col] = '—' if v is None else v
    rows_iv.append(row)

for name, bits_label in [('llama2_7b_4bit', 4), ('llama2_7b_2bit', 2)]:
    if name in llama2_lmeval_results:
        eva_vals = build_eva_row(llama2_lmeval_results[name], TASK_KEY_MAP_IV)
        row = {'Method': '*EVA*', 'Bits': bits_label}
        for col, v in zip(TABLE_IV_COLS, eva_vals):
            row[col] = v
        rows_iv.append(row)

# Sort: FP16 first, then 4-bit, then 2-bit
order = {16: 0, 4: 1, 2: 2}
rows_iv.sort(key=lambda r: (order.get(r['Bits'], 9), r['Method']))

df4 = pd.DataFrame(rows_iv)
display(Markdown('### TABLE IV — Accuracy (%) Comparison of LLaMA-2-7B on Downstream Benchmark'))
display(Markdown('*EVA rows are reproduced; all other values from published results.*'))
display(Markdown(df4.to_markdown(index=False)))

### TABLE X — MoE Models Downstream Accuracy

In [ ]:
moe_downstream_names = ['mixtral_4bit', 'mixtral_2bit', 'qwen3_4bit', 'qwen3_2bit']

moe_lmeval_results = {}
for name in moe_downstream_names:
    r = load_lmeval_result(name)
    if r:
        moe_lmeval_results[name] = r

In [ ]:
TASK_KEY_MAP_X = {
    'ARC-C': 'arc_challenge', 'ARC-E': 'arc_easy',
    'PIQA': 'piqa', 'BoolQ': 'boolq',
    'Winogrande': 'winogrande',
}


def build_moe_table(model_label, baselines, eva_results_map):
    rows = []
    for (method, bits), vals in baselines.items():
        row = {'Method': method, 'Bits': bits}
        for col, v in zip(TABLE_X_COLS, vals):
            row[col] = '—' if v is None else v
        rows.append(row)

    for name, bits_label, method_label in eva_results_map:
        if name in moe_lmeval_results:
            eva_vals = build_eva_row(moe_lmeval_results[name], TASK_KEY_MAP_X)
            row = {'Method': f'*{method_label}*', 'Bits': bits_label}
            for col, v in zip(TABLE_X_COLS, eva_vals):
                row[col] = v
            rows.append(row)

    order = {16: 0, 4: 1, 2: 2}
    rows.sort(key=lambda r: (order.get(r['Bits'], 9), r['Method']))
    return pd.DataFrame(rows)


display(Markdown('### TABLE X — Accuracy (%) Comparison of Mixture of Experts (MoE) Models'))
display(Markdown('*AQLM-4×8 and AQLM-2×8 rows are reproduced; all other values from published results.*'))

display(Markdown('#### Mixtral-8x7B'))
df_mixtral = build_moe_table(
    'Mixtral-8x7B',
    TABLE_X_MIXTRAL_BASELINES,
    [('mixtral_4bit', 4, 'AQLM-4×8'), ('mixtral_2bit', 2, 'AQLM-2×8')],
)
display(Markdown(df_mixtral.to_markdown(index=False)))

display(Markdown('#### Qwen3-30B-A3B'))
df_qwen = build_moe_table(
    'Qwen3-30B-A3B',
    TABLE_X_QWEN_BASELINES,
    [('qwen3_4bit', 4, 'AQLM-4×8'), ('qwen3_2bit', 2, 'AQLM-2×8')],
)
display(Markdown(df_qwen.to_markdown(index=False)))